In [29]:
import json                 # read the trace files (each one is a JSON document)
from pathlib import Path    # build paths and list files: Path(...).glob('trace_*.json')
import pandas as pd         # the per-step tables

In [30]:
trace_dirs = {'31b': Path('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces'),
               '12b': Path('run_12b/results/gemma-4-12b-it-qat-w4a16-ct/traces')}

for model in trace_dirs:
    total_files = 0
    folder = trace_dirs[model]
    json_files = sorted(folder.glob('trace_*.json'))
    total_files += len(json_files)
    print(json_files)

    print(f"Total Files in {folder.parts[2]}: {total_files}")

[WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_fastapi_13537.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_fastapi_13713.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_fastapi_14479.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_fastapi_14605.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_fastapi_15785.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_requests_7427.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_rich_2725.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_rich_3296.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_rich_3472.json'), WindowsPath('run_31b/results/gemma-4-31b-it-qat-w4a16-ct/traces/trace_rich_3521.json')]
Total Files in gemma-4-31b-it-qat-w4a16-ct: 10
[WindowsPath('run_12b/results/gemma-4-12b-it-qat

In [31]:
list(trace_dirs['31b'].glob('trace_rich_2725.json'))[0].stem.removeprefix('trace_')

'rich_2725'

In [32]:
p = trace_dirs['12b'] / 'trace_fastapi_13537.json'

# # 1. open the file, then load it
# with open(p, encoding='utf-8') as f:
#     trace = json.load(f)

# 2. read the text yourself, then parse the string
trace = json.loads(p.read_text(encoding='utf-8'))

In [33]:
# tr = json.loads(list(trace_dirs['31b'].glob('trace_rich_2725.json'))[0].read_text(encoding='utf-8'))
# tr

In [34]:
model_traces = [s for s in trace['steps'] if s['source'] == 'agent']

len(model_traces)

23

In [35]:
rows = []

for step in model_traces:
    tools = ', '.join([c.get('function_name') for c in step.get('tool_calls', [])])   # ['read_file']
    
    obs = step.get('observation')
    output_char = len(obs.get('content') or '') if obs else 0

    rows.append({'step_id': step.get('step_id'),
                'agent': step.get('model_name'),
                'elapsed_s': step.get('extra').get('elapsed_s'),
                'prompt_tokens': step.get('metrics', {}).get('prompt_tokens'),
                'completion_tokens': step.get('metrics', {}).get('completion_tokens'),
                'tools': tools,
                'output_chars': output_char
    })

df = pd.DataFrame(rows)

df.head()

,step_id,agent,elapsed_s,prompt_tokens,completion_tokens,tools,output_chars
0,3,/kaggle/input/models/google/gemma-4/other/gemm...,8.545966,3956.0,189.0,,0
1,4,/kaggle/input/models/google/gemma-4/other/gemm...,8.545966,NaN,NaN,,0
2,5,/kaggle/input/models/google/gemma-4/other/gemm...,8.545966,NaN,NaN,read_file,281
3,6,/kaggle/input/models/google/gemma-4/other/gemm...,10.320416,4246.0,123.0,run_command,241
4,7,/kaggle/input/models/google/gemma-4/other/gemm...,12.615509,4459.0,152.0,read_file,4952


In [36]:
print(df.shape)

df['tools'].value_counts()

(23, 7)


tools
read_file      11
run_command     8
                4
Name: count, dtype: int64

In [37]:
def trace_to_df(path, model):
    json_paths = sorted(path.glob('trace_*.json'))

    rows = []

    for p in json_paths:
        trace = json.loads(p.read_text(encoding='utf-8'))

        model_traces = [s for s in trace['steps'] if s['source'] == 'agent']

        for step in model_traces:
            tools = ', '.join(c.get('function_name') for c in step.get('tool_calls', []))   # ['read_file']
            
            obs = step.get('observation')
            output_char = len(obs.get('content') or '') if obs else 0

            rows.append({'model': model,
                        'task': p.stem.removeprefix('trace_'),
                        'step_id': step.get('step_id'),
                        'agent': step.get('extra', {}).get('author'),
                        'elapsed_s': step.get('extra').get('elapsed_s'),
                        'prompt_tokens': step.get('metrics', {}).get('prompt_tokens'),
                        'completion_tokens': step.get('metrics', {}).get('completion_tokens'),
                        'tools': tools,
                        'output_chars': output_char
            })

    df = pd.DataFrame(rows)

    return df

In [38]:
list_of_df = []

for model in trace_dirs:
    path = trace_dirs[model]
    df = trace_to_df(path, model)
    list_of_df.append(df)

df_steps = pd.concat(list_of_df, ignore_index=True)

In [39]:
df_steps

,model,task,step_id,agent,elapsed_s,prompt_tokens,completion_tokens,tools,output_chars
0,31b,fastapi_13537,3,swe_baseline_agent,17.330444,3956.0,325.0,run_command,237
1,31b,fastapi_13537,4,swe_baseline_agent,19.678286,4371.0,71.0,run_command,404
2,31b,fastapi_13537,5,swe_baseline_agent,21.325689,4585.0,47.0,read_file,4952
3,31b,fastapi_13537,6,swe_baseline_agent,24.728762,6003.0,76.0,run_command,425
4,31b,fastapi_13537,7,swe_baseline_agent,27.026543,6246.0,66.0,read_file,5815
...,...,...,...,...,...,...,...,...,...
512,12b,rich_3521,18,swe_baseline_agent,215.907877,NaN,NaN,,0
513,12b,rich_3521,19,swe_baseline_agent,215.907877,NaN,NaN,run_command,91
514,12b,rich_3521,20,swe_baseline_agent,280.470778,13969.0,4149.0,,0
515,12b,rich_3521,21,swe_baseline_agent,280.470778,NaN,NaN,,0


In [40]:
df_steps.groupby('model')['task'].nunique()

model
12b    10
31b    10
Name: task, dtype: int64

In [41]:
df_steps['agent'].value_counts()

agent
swe_baseline_agent     505
code_analyzer_agent     12
Name: count, dtype: int64

In [42]:
df_main_agent = df_steps[df_steps['agent'] == 'swe_baseline_agent'].sort_values(by=['model', 'task', 'step_id']).copy()

df_main_agent = df_main_agent.dropna(subset=['prompt_tokens'])

df_main_agent['prompt_diff'] = df_main_agent.groupby(['model', 'task'])['prompt_tokens'].diff()

df_main_agent.head()

,model,task,step_id,agent,elapsed_s,prompt_tokens,completion_tokens,tools,output_chars,prompt_diff
250,12b,fastapi_13537,3,swe_baseline_agent,8.545966,3956.0,189.0,,0,NaN
253,12b,fastapi_13537,6,swe_baseline_agent,10.320416,4246.0,123.0,run_command,241,290.0
254,12b,fastapi_13537,7,swe_baseline_agent,12.615509,4459.0,152.0,read_file,4952,213.0
255,12b,fastapi_13537,8,swe_baseline_agent,14.327223,5982.0,85.0,run_command,123,1523.0
256,12b,fastapi_13537,9,swe_baseline_agent,15.927475,6121.0,98.0,read_file,4952,139.0


In [43]:
df_drops = df_main_agent[df_main_agent['prompt_diff'] < 0]
df_drops[['model', 'task', 'step_id', 'prompt_tokens', 'prompt_diff']]

,model,task,step_id,prompt_tokens,prompt_diff
264,12b,fastapi_13537,17,10869.0,-4700.0
270,12b,fastapi_13537,24,5037.0,-7367.0
282,12b,fastapi_13713,12,7811.0,-8069.0
287,12b,fastapi_13713,17,7441.0,-7166.0
293,12b,fastapi_13713,23,7820.0,-7574.0
301,12b,fastapi_13713,31,6726.0,-8595.0
335,12b,fastapi_14479,16,5994.0,-9157.0
347,12b,fastapi_14479,28,5154.0,-9560.0
376,12b,fastapi_15785,14,8118.0,-7947.0
381,12b,fastapi_15785,19,6069.0,-9136.0


In [44]:
df_drops.shape

(35, 10)

In [45]:
# Tokens in each call: what the model read plus what it wrote
df_main_agent['total_tokens'] = df_main_agent['prompt_tokens'] + df_main_agent['completion_tokens']

# Group by task so values never cross from one task into the next
g = df_main_agent.groupby(['model', 'task'])

df_main_agent['prev_total'] = g['total_tokens'].shift()   # previous call's total, in the same task
df_main_agent['gap_s'] = g['elapsed_s'].diff()            # seconds since the previous call
df_main_agent['is_drop'] = df_main_agent['prompt_diff'] < 0

# Rebuild drops so it has the new columns (the old one was made before they existed)
drops = df_main_agent[df_main_agent['is_drop']]

In [46]:
# 1. Was the previous call always over the threshold before a drop?
print('all drops after >= 14336:', (drops['prev_total'] >= 14336).all())
print(drops['prev_total'].describe())

# And the other way round: calls over the threshold that were NOT followed by a drop
over_no_drop = df_main_agent[(~df_main_agent['is_drop']) & (df_main_agent['prev_total'] >= 14336)]
print('over threshold, no drop:', len(over_no_drop))

# 2. Are drop steps slower than normal steps?
df_main_agent.groupby('is_drop')['gap_s'].describe()

all drops after >= 14336: True
count       35.000000
mean     15524.371429
std        990.982729
min      14427.000000
25%      14739.000000
50%      15325.000000
75%      15878.500000
max      18392.000000
Name: prev_total, dtype: float64
over threshold, no drop: 10


,count,mean,std,min,25%,50%,75%,max
is_drop,,,,,,,,
False,426.0,8.118401,15.548949,0.625560,1.649256,2.670875,5.895835,131.477915
True,35.0,21.872209,9.143466,9.659348,14.831359,19.685367,27.549092,52.607675


## Failure taxonomy

One row per task: how it ended and why. Built from three sources:
- `per_task.csv` (the notebook's own summary): solved or not, end reason, patch size.
- The traces: one row per tool call, with the tool's result status and error type.
- `tasks.jsonl`: the files the correct fix (`patch`) changes, to check whether the agent edited the right file.

Facts checked in these traces: each agent step has at most one tool call, and its result is in `observation.content` as a JSON string
(`{"status": "ok" | "error", "error_type": ..., ...}`). Gold patches name changed files on `+++ b/<path>` lines (no `diff --git` lines).
No task edited files through `run_command` (no `sed -i`, redirects etc.), so `edit_file` / `write_file` cover all edits here.

In [47]:
import re

# Step 1: the per-task table from both runs (results folder = parent of the traces folder)
df_tasks = pd.concat([pd.read_csv(trace_dirs[m].parent / 'per_task.csv').assign(model=m) for m in trace_dirs],
                     ignore_index=True)

df_tasks[['model', 'task_id', 'resolved', 'end_reason', 'tool_calls', 'patch_chars']]

,model,task_id,resolved,end_reason,tool_calls,patch_chars
0,31b,rich_3521,False,out_of_time,13,0
1,31b,rich_2725,True,submitted,22,2344
2,31b,rich_3296,False,out_of_time,29,503
3,31b,rich_3472,False,out_of_time,30,0
4,31b,fastapi_15785,False,submitted,30,0
5,31b,fastapi_13713,False,submitted,30,1377
6,31b,fastapi_14479,False,submitted,11,521
7,31b,fastapi_13537,True,submitted,22,2888
8,31b,fastapi_14605,False,submitted,30,4898
9,31b,requests_7427,False,out_of_time,19,2629


In [48]:
# Step 2: one row per tool call (main agent only)
def tool_calls_to_df(path, model):
    rows = []
    for p in sorted(path.glob('trace_*.json')):
        trace = json.loads(p.read_text(encoding='utf-8'))
        steps = [s for s in trace['steps']
                 if s['source'] == 'agent' and s.get('tool_calls')
                 and s.get('extra', {}).get('author') == 'swe_baseline_agent']

        for call_no, step in enumerate(steps, start=1):
            call = step['tool_calls'][0]               # at most one tool call per step (checked)
            args = call.get('arguments') or {}
            obs = step.get('observation')              # 2 calls have none (task ended first)
            result = json.loads(obs['content']) if obs and obs.get('content') else {}

            rows.append({'model': model,
                         'task': p.stem.removeprefix('trace_'),
                         'call_no': call_no,
                         'tool': call['function_name'],
                         'filepath': args.get('filepath'),
                         'bad_args': any('"' in k for k in args),   # e.g. 'start_line"'
                         'status': result.get('status'),
                         'error_type': result.get('error_type'),
                         'output_chars': len(obs['content']) if obs else 0})
    return pd.DataFrame(rows)

df_calls = pd.concat([tool_calls_to_df(trace_dirs[m], m) for m in trace_dirs], ignore_index=True)

print(df_calls.shape)
df_calls['error_type'].value_counts()

(474, 9)


error_type
FileEditError     18
CommandError      17
BudgetExceeded     5
ValueError         1
Name: count, dtype: int64

In [49]:
df_calls.head()

,model,task,call_no,tool,filepath,bad_args,status,error_type,output_chars
0,31b,fastapi_13537,1,run_command,NaN,False,ok,NaN,237
1,31b,fastapi_13537,2,run_command,NaN,False,ok,NaN,404
2,31b,fastapi_13537,3,read_file,fastapi/dependencies/utils.py,False,ok,NaN,4952
3,31b,fastapi_13537,4,run_command,NaN,False,ok,NaN,425
4,31b,fastapi_13537,5,read_file,fastapi/dependencies/utils.py,False,ok,NaN,5815


In [50]:
# Step 3a: gold files per task, from the correct fix in tasks.jsonl
gold_files = {}
with open('../../competition_data/tasks.jsonl', encoding='utf-8') as f:
    for line in f:
        t = json.loads(line)
        gold_files[t['instance_id']] = set(re.findall(r'^\+\+\+ b/(\S+)', t['patch'], flags=re.M))

{k: gold_files[k] for k in df_tasks['task_id'].unique()}

{'rich_3521': {'rich/segment.py'},
 'rich_2725': {'rich/table.py'},
 'rich_3296': {'rich/syntax.py'},
 'rich_3472': {'rich/pretty.py'},
 'fastapi_15785': {'fastapi/openapi/utils.py', 'fastapi/routing.py'},
 'fastapi_13713': {'fastapi/applications.py', 'fastapi/openapi/utils.py'},
 'fastapi_14479': {'fastapi/dependencies/utils.py'},
 'fastapi_13537': {'fastapi/dependencies/utils.py'},
 'fastapi_14605': {'fastapi/dependencies/utils.py',
  'fastapi/exceptions.py',
  'fastapi/openapi/utils.py',
  'fastapi/params.py',
  'fastapi/routing.py',
  'fastapi/temp_pydantic_v1_params.py',
  'fastapi/utils.py'},
 'requests_7427': {'src/requests/utils.py'}}

In [51]:
# Step 3b: summarise each task's tool calls
def summarise(calls):
    edits = calls[calls['tool'].isin(['edit_file', 'write_file']) & (calls['status'] == 'ok')]
    edited = set(edits['filepath'])
    gold = gold_files[calls.name[1]]               # calls.name = (model, task)
    return pd.Series({
        'n_calls': len(calls),                     # includes submit_patch / get_status, so can pass 30
        'n_errors': (calls['status'] == 'error').sum(),
        'n_budget_errors': (calls['error_type'] == 'BudgetExceeded').sum(),
        'n_edit_fails': (calls['error_type'] == 'FileEditError').sum(),
        'n_bad_args': calls['bad_args'].sum(),
        'first_edit_call': edits['call_no'].min() if len(edits) else None,
        'edited_files': sorted(edited),
        'touched_gold': bool(edited & gold),
        'edited_tests': any(f.startswith('tests/') for f in edited),
    })

df_task_calls = df_calls.groupby(['model', 'task']).apply(summarise, include_groups=False).reset_index()

# Context use per task, from df_main_agent (the compaction cells above)
df_context = (df_main_agent.groupby(['model', 'task'])
              .agg(peak_context=('total_tokens', 'max'), n_compactions=('is_drop', 'sum'))
              .reset_index())

df_tax = (df_tasks.rename(columns={'task_id': 'task'})
          .merge(df_task_calls, on=['model', 'task'], how='left')
          .merge(df_context, on=['model', 'task'], how='left'))

In [52]:
df_task_calls['touched_gold'].value_counts()

touched_gold
False    10
True     10
Name: count, dtype: int64

In [53]:
df_task_calls

,model,task,n_calls,n_errors,n_budget_errors,n_edit_fails,n_bad_args,first_edit_call,edited_files,touched_gold,edited_tests
0,12b,fastapi_13537,19,0,0,0,0,NaN,[],False,False
1,12b,fastapi_13713,37,4,4,0,0,23.0,[fastapi/openapi/utils.py],True,False
2,12b,fastapi_14479,26,7,0,5,0,NaN,[],False,False
3,12b,fastapi_14605,15,1,0,0,0,8.0,"[fastapi/temp_pydantic_v1_params.py, fastapi/w...",True,False
4,12b,fastapi_15785,30,1,0,0,0,NaN,[],False,False
5,12b,requests_7427,24,2,0,2,0,6.0,[src/requests/utils.py],True,False
6,12b,rich_2725,24,9,0,9,0,NaN,[],False,False
7,12b,rich_3296,13,1,0,0,0,NaN,[],False,False
8,12b,rich_3472,32,1,1,0,0,NaN,[],False,False
9,12b,rich_3521,12,0,0,0,0,NaN,[],False,False


In [54]:
# Step 4: one label per task; the first rule that matches wins
def label(r):
    if r['resolved']:
        return 'solved'
    if r['end_reason'] == 'context_full':
        return 'context overflow'
    if r['end_reason'] == 'crashed':
        return 'crashed'
    if not r['edited_files']:
        return 'edits all failed' if r['n_edit_fails'] > 0 else 'never edited'
    if not r['touched_gold']:
        return 'wrong file'
    if r['end_reason'] == 'submitted':
        return 'wrong fix'
    return 'right file, out of budget'

df_tax['label'] = df_tax.apply(label, axis=1)

df_tax[['model', 'task', 'label', 'end_reason', 'n_calls', 'first_edit_call', 'n_errors', 'n_bad_args',
        'edited_tests', 'peak_context', 'n_compactions', 'edited_files']].sort_values(['model', 'label'])

,model,task,label,end_reason,n_calls,first_edit_call,n_errors,n_bad_args,edited_tests,peak_context,n_compactions,edited_files
10,12b,rich_3521,context overflow,context_full,12,NaN,0,0,False,18118.0,1,[]
11,12b,rich_2725,context overflow,context_full,24,NaN,9,0,False,18211.0,1,[]
12,12b,rich_3296,context overflow,context_full,13,NaN,1,0,False,17029.0,1,[]
16,12b,fastapi_14479,edits all failed,out_of_time,26,NaN,7,0,False,15273.0,2,[]
13,12b,rich_3472,never edited,submitted,32,NaN,1,0,False,16241.0,4,[]
14,12b,fastapi_15785,never edited,out_of_tool_calls,30,NaN,1,0,False,16115.0,3,[]
17,12b,fastapi_13537,never edited,out_of_time,19,NaN,0,0,False,18392.0,2,[]
15,12b,fastapi_13713,"right file, out of budget",out_of_time,37,23.0,4,0,False,15925.0,4,[fastapi/openapi/utils.py]
18,12b,fastapi_14605,"right file, out of budget",out_of_time,15,8.0,1,0,False,13030.0,0,"[fastapi/temp_pydantic_v1_params.py, fastapi/w..."
19,12b,requests_7427,"right file, out of budget",out_of_time,24,6.0,2,0,False,18277.0,4,[src/requests/utils.py]


In [55]:
pd.crosstab(df_tax['label'], df_tax['model'], margins=True)

model,12b,31b,All
label,,,
context overflow,3,0,3
edits all failed,1,0,1
never edited,3,3,6
"right file, out of budget",3,2,5
solved,0,2,2
wrong fix,0,3,3
All,10,10,20


In [56]:
# Which tools fill the context? Output size of each tool's results, both models together
(df_calls.groupby('tool')['output_chars']
 .agg(calls='count', median_chars='median', total_chars='sum')
 .assign(share_of_total=lambda d: (d['total_chars'] / d['total_chars'].sum()).round(2))
 .sort_values('total_chars', ascending=False))

,calls,median_chars,total_chars,share_of_total
tool,,,,
read_file,246,5020.5,1106617,0.89
run_command,163,223.0,105861,0.09
edit_file,43,523.0,25224,0.02
get_status,3,295.0,881,0.00
write_file,10,60.0,601,0.00
submit_patch,6,55.5,329,0.00
get_code_neighbors,1,266.0,266,0.00
search_similar_code,1,109.0,109,0.00
code_analyzer_agent,1,0.0,0,0.00


## Taxonomy: the remaining pieces

Three items from the roadmap's failure list that the table above didn't cover yet.

**A. Tool-call format errors.** Besides the broken argument names (`n_bad_args`), the model sometimes writes a tool call as plain text
that the parser doesn't recognise (e.g. `call:read_file{...}<tool_call|>`). The harness then sees a text reply with no tool call and
sends a "please continue" nudge. It also leaks Gemma's channel markers (`<|channel>thought`, `<channel|>`) into visible text.

**B. Patch fails to apply vs tests fail.** Checked in the harness source:
- If the agent's patch doesn't apply, the result's `error` becomes `Failed to apply agent patch: ...` (`swegemma/harness/verification.py:474`).
- If the agent hit a limit with an empty patch, tests are skipped: `test_exit_code = -1` (`swegemma/evaluate.py:324`).
- Otherwise pytest runs on the task's test files; `resolved` = exit code 0 (`verification.py`, `pytest_cmd`). 1 = some tests failed,
  2 = pytest stopped early, which usually means a collection error such as an import of a name the patch didn't add (pytest's own codes;
  the test output itself wasn't saved, so the exact reason per task is not visible).
- Edits to protected test files are stripped from the patch before testing (`verification.py:88`, `strip_protected_files_from_patch`).

The per-task log files are empty, but `notebook.log` prints `exit_code` after each `Evaluating <task>` line.

**C. Context growth: reply vs tool output.** Fit the growth of the prompt between two consecutive main-agent calls (skipping compactions)
on the previous call's reply tokens and its tool-output characters.

In [62]:
# A. Format errors in the main agent's text replies, plus the harness's nudges
def format_events_to_df(path, model):
    rows = []
    for p in sorted(path.glob('trace_*.json')):
        trace = json.loads(p.read_text(encoding='utf-8'))
        for s in trace['steps']:
            ex = s.get('extra', {})
            msg = s.get('message') if isinstance(s.get('message'), str) else ''
            is_main_text = (s['source'] == 'agent' and ex.get('author') == 'swe_baseline_agent'
                            and not s.get('tool_calls') and ex.get('event_type') in ('text', 'final'))
            is_nudge = s['source'] == 'system' and ex.get('event_type') == 'continuation_nudge'
            if is_main_text or is_nudge:
                rows.append({'model': model,
                             'task': p.stem.removeprefix('trace_'),
                             'step_id': s['step_id'],
                             'event_type': ex.get('event_type'),
                             'unparsed_call': is_main_text and 'call:' in msg and '<tool_call|>' in msg,
                             'leaked_marker': is_main_text and ('<|channel>' in msg or '<channel|>' in msg),
                             'text': msg[:100]})
    return pd.DataFrame(rows)

df_format = pd.concat([format_events_to_df(trace_dirs[m], m) for m in trace_dirs], ignore_index=True)

df_format[df_format['unparsed_call'] | df_format['leaked_marker'] | (df_format['event_type'] == 'continuation_nudge')]

,model,task,step_id,event_type,unparsed_call,leaked_marker,text
4,12b,fastapi_13537,4,text,False,True,I will start by exploring the `fastapi/request...
5,12b,fastapi_13537,22,final,True,False,"call:read_file{filepath:<|""|>fastapi/dependenc..."
6,12b,fastapi_13537,23,continuation_nudge,False,False,Please continue your work using the available ...
7,12b,fastapi_14479,4,text,False,True,I will start by reading the `fastapi/dependenc...
8,12b,fastapi_15785,4,text,False,True,I will start by searching for the `APIRouter` ...
9,12b,fastapi_15785,36,final,False,True,<|channel>thought\nI'll use `search_similar_co...
10,12b,requests_7427,4,text,False,True,"I will start by searching for ""no_proxy"" in th..."
11,12b,rich_3472,35,text,False,True,<|channel>thought\n<channel|>I've analyzed the...
13,12b,rich_3521,4,text,False,True,I will begin by examining the `rich/segment.py...
14,12b,rich_3521,10,text,False,True,<|channel>thought\n<channel|>


In [63]:
# Per task: all format problems side by side (bad argument names and ValueErrors come from df_calls)
fmt = (df_format.groupby(['model', 'task'])
       .agg(n_unparsed_calls=('unparsed_call', 'sum'),
            n_leaked_markers=('leaked_marker', 'sum'),
            n_nudges=('event_type', lambda s: (s == 'continuation_nudge').sum()))
       .reset_index())
value_errors = (df_calls[df_calls['error_type'] == 'ValueError']
                .groupby(['model', 'task']).size().rename('n_value_errors').reset_index())

df_tax = (df_tax.drop(columns=['n_unparsed_calls', 'n_leaked_markers', 'n_nudges', 'n_value_errors'], errors='ignore')
          .merge(fmt, on=['model', 'task'], how='left')
          .merge(value_errors, on=['model', 'task'], how='left'))
fmt_cols = ['n_bad_args', 'n_value_errors', 'n_unparsed_calls', 'n_leaked_markers', 'n_nudges']
df_tax[fmt_cols] = df_tax[fmt_cols].fillna(0).astype(int)

print(df_tax.groupby('model')[fmt_cols].sum())
df_tax.loc[df_tax[fmt_cols].sum(axis=1) > 0, ['model', 'task', 'label'] + fmt_cols]

       n_bad_args  n_value_errors  n_unparsed_calls  n_leaked_markers  \
model                                                                   
12b             0               1                 1                10   
31b            10               0                 0                 0   

       n_nudges  
model            
12b           1  
31b           0  


,model,task,label,n_bad_args,n_value_errors,n_unparsed_calls,n_leaked_markers,n_nudges
2,31b,rich_3296,"right file, out of budget",1,0,0,0,0
5,31b,fastapi_13713,wrong fix,5,0,0,0,0
7,31b,fastapi_13537,solved,2,0,0,0,0
9,31b,requests_7427,"right file, out of budget",2,0,0,0,0
10,12b,rich_3521,context overflow,0,0,0,4,0
13,12b,rich_3472,never edited,0,0,0,1,0
14,12b,fastapi_15785,never edited,0,0,0,2,0
16,12b,fastapi_14479,edits all failed,0,1,0,1,0
17,12b,fastapi_13537,never edited,0,0,1,1,1
19,12b,requests_7427,"right file, out of budget",0,0,0,1,0


In [64]:
# B. Did any patch fail to apply? (the harness would put this text in `error`)
print('patches that failed to apply:', df_tax['error'].str.contains('Failed to apply', na=False).sum())

# Test exit code per task, from notebook.log: remember the task from 'Evaluating <task>', read the code from '-> resolved=...'
exit_rows = []
for m in trace_dirs:
    task = None
    for line in Path(f'run_{m}/notebook.log').read_text(encoding='utf-8').splitlines():
        if (hit := re.search(r'Evaluating (\S+) ', line)):
            task = hit.group(1)
        elif (hit := re.search(r'-> resolved=\w+, exit_code=(-?\d+)', line)):
            exit_rows.append({'model': m, 'task': task, 'test_exit_code': int(hit.group(1))})

exit_meaning = {-1: 'not run (limit hit, empty patch)', 0: 'passed', 1: 'ran, some failed',
                2: 'pytest stopped early (e.g. import error)'}
df_tax = (df_tax.drop(columns=['test_exit_code', 'test_result'], errors='ignore')
          .merge(pd.DataFrame(exit_rows), on=['model', 'task'], how='left'))
df_tax['test_result'] = df_tax['test_exit_code'].map(exit_meaning)

pd.crosstab([df_tax['label']], [df_tax['model'], df_tax['test_result']])

patches that failed to apply: 0


model                                                  12b  \
test_result               not run (limit hit, empty patch)   
label                                                        
context overflow                                         3   
edits all failed                                         1   
never edited                                             2   
right file, out of budget                                0   
solved                                                   0   
wrong fix                                                0   

model                                                               \
test_result               pytest stopped early (e.g. import error)   
label                                                                
context overflow                                                 0   
edits all failed                                                 0   
never edited                                                     0   
right file, out of budget                                        2   
solved                                                           0   
wrong fix                                                        0   

model                                                                   31b  \
test_result               ran, some failed not run (limit hit, empty patch)   
label                                                                         
context overflow                         0                                0   
edits all failed                         0                                0   
never edited                             1                                2   
right file, out of budget                1                                0   
solved                                   0                                0   
wrong fix                                0                                0   

model                                                                      \
test_result               passed pytest stopped early (e.g. import error)   
label                                                                       
context overflow               0                                        0   
edits all failed               0                                        0   
never edited                   0                                        1   
right file, out of budget      0                                        0   
solved                         2                                        0   
wrong fix                      0                                        2   

model                                       
test_result               ran, some failed  
label                                       
context overflow                         0  
edits all failed                         0  
never edited                             0  
right file, out of budget                2  
solved                                   0  
wrong fix                                1

In [65]:
import numpy as np

# C. One row per main-agent model call: tokens in, tokens out, tool output that followed
def model_calls_to_df(path, model):
    rows = []
    for p in sorted(path.glob('trace_*.json')):
        trace = json.loads(p.read_text(encoding='utf-8'))
        calls = []
        for s in trace['steps']:
            if s['source'] != 'agent' or s.get('extra', {}).get('author') != 'swe_baseline_agent':
                continue
            if s.get('metrics'):                       # first event of a new model call
                calls.append({'model': model, 'task': p.stem.removeprefix('trace_'),
                              'prompt_tokens': s['metrics']['prompt_tokens'],
                              'reply_tokens': s['metrics']['completion_tokens'],
                              'tool_chars': 0})
            if calls:                                  # later events of the same call carry the tool result
                obs = s.get('observation')
                calls[-1]['tool_chars'] += len(obs['content']) if obs and obs.get('content') else 0
        rows += calls
    return pd.DataFrame(rows)

df_mc = pd.concat([model_calls_to_df(trace_dirs[m], m) for m in trace_dirs], ignore_index=True)
df_mc['growth'] = df_mc.groupby(['model', 'task'])['prompt_tokens'].shift(-1) - df_mc['prompt_tokens']

# Fit growth = a * reply_tokens + b * tool_chars + c, skipping compactions (growth <= 0) and each task's last call
fit_rows = df_mc[df_mc['growth'] > 0]
X = np.c_[fit_rows['reply_tokens'], fit_rows['tool_chars'], np.ones(len(fit_rows))]
(a, b, c), *_ = np.linalg.lstsq(X, fit_rows['growth'], rcond=None)
r2 = 1 - ((fit_rows['growth'] - X @ [a, b, c]) ** 2).sum() / ((fit_rows['growth'] - fit_rows['growth'].mean()) ** 2).sum()
print(f'growth = {a:.2f} * reply_tokens + {b:.3f} * tool_chars + {c:.0f}   (R2 = {r2:.2f}, n = {len(fit_rows)})')
print(f'so ~{1 / b:.1f} characters of tool output per token, and {a:.0%} of each reply stays in the context')

growth = 1.00 * reply_tokens + 0.281 * tool_chars + 13   (R2 = 0.91, n = 426)
so ~3.6 characters of tool output per token, and 100% of each reply stays in the context


In [66]:
# Per task: how much of the conversation came from the model's replies vs from tool output
ctx = (df_mc.groupby(['model', 'task'])
       .agg(reply_tokens=('reply_tokens', 'sum'), tool_chars=('tool_chars', 'sum'))
       .assign(tool_tokens=lambda d: (d['tool_chars'] * b).round()))
ctx['tool_share'] = (ctx['tool_tokens'] / (ctx['tool_tokens'] + ctx['reply_tokens'])).round(2)

df_tax = df_tax.drop(columns=['reply_tokens', 'tool_tokens', 'tool_share'], errors='ignore').merge(
    ctx[['reply_tokens', 'tool_tokens', 'tool_share']].reset_index(), on=['model', 'task'], how='left')

print(ctx.groupby('model')[['reply_tokens', 'tool_tokens']].sum().assign(
    tool_share=lambda d: (d['tool_tokens'] / (d['tool_tokens'] + d['reply_tokens'])).round(2)))
df_tax[['model', 'task', 'label', 'reply_tokens', 'tool_tokens', 'tool_share', 'peak_context', 'n_compactions']].sort_values(['model', 'tool_share'])

       reply_tokens  tool_tokens  tool_share
model                                       
12b           97086     207733.0        0.68
31b           61402     141215.0        0.70


,model,task,label,reply_tokens,tool_tokens,tool_share,peak_context,n_compactions
10,12b,rich_3521,context overflow,17319,4366.0,0.20,18118.0,1
17,12b,fastapi_13537,never edited,15960,8281.0,0.34,18392.0,2
12,12b,rich_3296,context overflow,12048,12615.0,0.51,17029.0,1
11,12b,rich_2725,context overflow,10144,14319.0,0.59,18211.0,1
19,12b,requests_7427,"right file, out of budget",12838,25585.0,0.67,18277.0,4
18,12b,fastapi_14605,"right file, out of budget",2628,6902.0,0.72,13030.0,0
16,12b,fastapi_14479,edits all failed,5656,17157.0,0.75,15273.0,2
13,12b,rich_3472,never edited,10506,35301.0,0.77,16241.0,4
14,12b,fastapi_15785,never edited,5038,37119.0,0.88,16115.0,3
15,12b,fastapi_13713,"right file, out of budget",4949,46088.0,0.90,15925.0,4
